# Gold — Sucesso de entregas em feriados

Desenvolvido por: Ygor Moraes

## Objetivo

Criar a Gold `gold_ecommerce_rastreamento_entregas_feriados_sucesso`, comparando a taxa de sucesso de eventos de entrega em feriados e fora de feriados.

## Regra de negócio

Cruzar a data do evento de rastreamento com a Silver de feriados.

A Gold calcula:

- quantidade de eventos por mês;
- quantidade de eventos entregues;
- quantidade de eventos não entregues;
- quantidade de entregas em feriado;
- taxa de sucesso percentual;
- flag `evento_em_feriado`.

## Fontes

- Silver `ecommerce_rastreamento_entregas`
- Silver `feriados`

## Cuidados técnicos

- `ecommerce_rastreamento_entregas` é lida como Delta.
- `feriados` é lida como Delta.
- A Silver de feriados deve ter uma linha por `data_feriado`.
- O join por data não pode inflar a quantidade de eventos.
- A regra atual considera apenas feriados nacionais previstos em lei.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa funções e define parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    concat_ws,
    count,
    current_timestamp,
    lpad,
    lit,
    month,
    round as spark_round,
    sum as spark_sum,
    to_date,
    when,
    year
)

SILVER_RASTREAMENTO_TABLE = "ecommerce_rastreamento_entregas"
SILVER_FERIADOS_TABLE = "feriados"

SILVER_RASTREAMENTO_PATH = f"{SILVER_BASE_PATH}{SILVER_RASTREAMENTO_TABLE}"
SILVER_FERIADOS_PATH = f"{SILVER_BASE_PATH}{SILVER_FERIADOS_TABLE}"

GOLD_TABLE = "gold_ecommerce_rastreamento_entregas_feriados_sucesso"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

RASTREAMENTO_REQUIRED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "status_entrega",
    "dt_evento"
]

FERIADOS_REQUIRED_COLUMNS = [
    "data_feriado",
    "nome_feriado",
    "tipo_feriado",
    "categoria_feriado",
    "ano",
    "mes",
    "silver_processed_at"
]

GOLD_KEY_COLUMNS = [
    "ano_evento",
    "mes_evento",
    "evento_em_feriado"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_RASTREAMENTO_PATH:", SILVER_RASTREAMENTO_PATH)
print("SILVER_FERIADOS_PATH:", SILVER_FERIADOS_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers necessárias para montar a Gold.

df_rastreamento = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_RASTREAMENTO_PATH)
)

df_feriados = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_FERIADOS_PATH)
)

total_rastreamento = df_rastreamento.count()
total_feriados = df_feriados.count()

print("Silver de rastreamento lida com sucesso.")
print(f"Total eventos rastreamento: {total_rastreamento}")

print("Silver de feriados lida com sucesso.")
print(f"Total feriados: {total_feriados}")

In [0]:
# Valida colunas obrigatórias, chaves e datas das fontes.

validate_required_columns(df_rastreamento, RASTREAMENTO_REQUIRED_COLUMNS)
validate_required_columns(df_feriados, FERIADOS_REQUIRED_COLUMNS)

rastreamento_distintos = (
    df_rastreamento
    .select(col("id_rastreamento").cast("int").alias("id_rastreamento"))
    .distinct()
    .count()
)

rastreamento_duplicados = total_rastreamento - rastreamento_distintos

eventos_sem_dt_evento = (
    df_rastreamento
    .filter(col("dt_evento").isNull())
    .count()
)

eventos_sem_status = (
    df_rastreamento
    .filter(col("status_entrega").isNull())
    .count()
)

datas_feriado_distintas = (
    df_feriados
    .select(to_date(col("data_feriado")).alias("data_feriado"))
    .distinct()
    .count()
)

datas_feriado_duplicadas = total_feriados - datas_feriado_distintas

feriados_sem_data = (
    df_feriados
    .filter(col("data_feriado").isNull())
    .count()
)

print(f"Total eventos rastreamento: {total_rastreamento}")
print(f"Eventos distintos por id_rastreamento: {rastreamento_distintos}")
print(f"Eventos duplicados por id_rastreamento: {rastreamento_duplicados}")
print(f"Eventos sem dt_evento: {eventos_sem_dt_evento}")
print(f"Eventos sem status_entrega: {eventos_sem_status}")

print(f"Total feriados: {total_feriados}")
print(f"Datas de feriado distintas: {datas_feriado_distintas}")
print(f"Datas de feriado duplicadas: {datas_feriado_duplicadas}")
print(f"Feriados sem data_feriado: {feriados_sem_data}")

if rastreamento_duplicados > 0:
    raise Exception("Erro: existem eventos duplicados por id_rastreamento.")

if eventos_sem_dt_evento > 0:
    raise Exception("Erro: existem eventos sem dt_evento.")

if eventos_sem_status > 0:
    raise Exception("Erro: existem eventos sem status_entrega.")

if datas_feriado_duplicadas > 0:
    raise Exception("Erro: existem datas duplicadas na Silver de feriados.")

if feriados_sem_data > 0:
    raise Exception("Erro: existem feriados sem data_feriado.")

print("Validação OK: fontes mínimas conferidas.")

In [0]:
# Prepara as bases de rastreamento e feriados para o join por data.

df_rastreamento_base = (
    df_rastreamento
    .select(
        col("id_rastreamento").cast("int").alias("id_rastreamento"),
        col("id_pedido_ecommerce").cast("int").alias("id_pedido_ecommerce"),
        col("status_entrega"),
        col("dt_evento"),
        to_date(col("dt_evento")).alias("data_evento")
    )
    .withColumn("ano_evento", year(col("dt_evento")))
    .withColumn("mes_evento", month(col("dt_evento")))
)

df_feriados_base = (
    df_feriados
    .select(
        to_date(col("data_feriado")).alias("data_evento")
    )
    .distinct()
)

print("Bases preparadas para join.")

In [0]:
# Cria flag indicando se o evento ocorreu em feriado.

df_eventos_feriados_base = (
    df_rastreamento_base
    .join(
        df_feriados_base.withColumn("evento_em_feriado", lit(1)),
        on="data_evento",
        how="left"
    )
    .withColumn(
        "evento_em_feriado",
        when(col("evento_em_feriado").isNotNull(), 1).otherwise(0)
    )
)

print("Base com flag de feriado criada.")

In [0]:
# Valida se o join com feriados não inflou os eventos.

total_eventos_feriados_base = df_eventos_feriados_base.count()

eventos_distintos_feriados_base = (
    df_eventos_feriados_base
    .select("id_rastreamento")
    .distinct()
    .count()
)

eventos_duplicados_feriados_base = total_eventos_feriados_base - eventos_distintos_feriados_base

eventos_sem_data = (
    df_eventos_feriados_base
    .filter(col("data_evento").isNull())
    .count()
)

print(f"Total eventos original: {total_rastreamento}")
print(f"Total eventos após join: {total_eventos_feriados_base}")
print(f"Eventos distintos após join: {eventos_distintos_feriados_base}")
print(f"Eventos duplicados após join: {eventos_duplicados_feriados_base}")
print(f"Eventos sem data_evento: {eventos_sem_data}")

if total_eventos_feriados_base != total_rastreamento:
    raise Exception("Erro: o join com feriados alterou a quantidade de eventos.")

if eventos_duplicados_feriados_base > 0:
    raise Exception("Erro: o join com feriados gerou eventos duplicados.")

if eventos_sem_data > 0:
    raise Exception("Erro: existem eventos sem data_evento após o join.")

print("Validação OK: join com feriados manteve um registro por evento.")

In [0]:
# Cria a Gold de sucesso de entregas em feriados.

df_gold = (
    df_eventos_feriados_base
    .groupBy(
        "ano_evento",
        "mes_evento",
        "evento_em_feriado"
    )
    .agg(
        count("id_rastreamento").alias("qtd_eventos"),
        spark_sum(
            when(col("status_entrega") == "entregue", 1).otherwise(0)
        ).alias("qtd_eventos_entregues")
    )
    .withColumn(
        "qtd_eventos_nao_entregues",
        col("qtd_eventos") - col("qtd_eventos_entregues")
    )
    .withColumn(
        "taxa_sucesso_percentual",
        spark_round((col("qtd_eventos_entregues") / col("qtd_eventos")) * 100, 2)
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("ano_evento", "mes_evento", "evento_em_feriado")
)

print("Gold de sucesso em feriados criada em memória.")
display(df_gold)

In [0]:
# Valida totais, chaves e campos principais da Gold.

total_linhas_gold = df_gold.count()

total_chaves_gold = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_gold = total_linhas_gold - total_chaves_gold

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_eventos").alias("total_eventos_gold"),
        spark_sum("qtd_eventos_entregues").alias("total_eventos_entregues"),
        spark_sum("qtd_eventos_nao_entregues").alias("total_eventos_nao_entregues")
    )
    .collect()[0]
)

nulos_gold = (
    df_gold
    .filter(
        col("ano_evento").isNull() |
        col("mes_evento").isNull() |
        col("evento_em_feriado").isNull() |
        col("qtd_eventos").isNull() |
        col("qtd_eventos_entregues").isNull() |
        col("qtd_eventos_nao_entregues").isNull() |
        col("taxa_sucesso_percentual").isNull()
    )
    .count()
)

print(f"Total eventos fonte: {total_rastreamento}")
print(f"Total eventos na Gold: {validacao_gold['total_eventos_gold']}")
print(f"Eventos entregues na Gold: {validacao_gold['total_eventos_entregues']}")
print(f"Eventos não entregues na Gold: {validacao_gold['total_eventos_nao_entregues']}")
print(f"Total linhas Gold: {total_linhas_gold}")
print(f"Chaves duplicadas Gold: {chaves_duplicadas_gold}")
print(f"Linhas com nulos principais: {nulos_gold}")

if validacao_gold["total_eventos_gold"] != total_rastreamento:
    raise Exception("Erro: total de eventos da Gold não fecha com a fonte.")

if (
    validacao_gold["total_eventos_entregues"] +
    validacao_gold["total_eventos_nao_entregues"]
    != validacao_gold["total_eventos_gold"]
):
    raise Exception("Erro: entregues + não entregues não fecha com total de eventos.")

if chaves_duplicadas_gold > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold.")

if nulos_gold > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold.")

print("Validação OK: Gold em memória conferida.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .partitionBy("ano_evento", "mes_evento")
    .save(GOLD_PATH)
)

print(f"Gold gravada com sucesso em Delta: {GOLD_PATH}")

In [0]:
# Lê e valida a Gold Delta gravada.

df_gold_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

total_linhas_gold_saved = df_gold_saved.count()

total_chaves_gold_saved = (
    df_gold_saved
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_gold_saved = total_linhas_gold_saved - total_chaves_gold_saved

validacao_gold_saved = (
    df_gold_saved
    .agg(
        spark_sum("qtd_eventos").alias("total_eventos_gold"),
        spark_sum("qtd_eventos_entregues").alias("total_eventos_entregues"),
        spark_sum("qtd_eventos_nao_entregues").alias("total_eventos_nao_entregues")
    )
    .collect()[0]
)

nulos_gold_saved = (
    df_gold_saved
    .filter(
        col("ano_evento").isNull() |
        col("mes_evento").isNull() |
        col("evento_em_feriado").isNull() |
        col("qtd_eventos").isNull() |
        col("qtd_eventos_entregues").isNull() |
        col("qtd_eventos_nao_entregues").isNull() |
        col("taxa_sucesso_percentual").isNull()
    )
    .count()
)

print(f"Total linhas Gold Delta: {total_linhas_gold_saved}")
print(f"Chaves duplicadas Gold Delta: {chaves_duplicadas_gold_saved}")
print(f"Total eventos fonte: {total_rastreamento}")
print(f"Total eventos Gold Delta: {validacao_gold_saved['total_eventos_gold']}")
print(f"Eventos entregues Gold Delta: {validacao_gold_saved['total_eventos_entregues']}")
print(f"Eventos não entregues Gold Delta: {validacao_gold_saved['total_eventos_nao_entregues']}")
print(f"Linhas com nulos principais Gold Delta: {nulos_gold_saved}")

if validacao_gold_saved["total_eventos_gold"] != total_rastreamento:
    raise Exception("Erro: total de eventos da Gold Delta não confere.")

if (
    validacao_gold_saved["total_eventos_entregues"] +
    validacao_gold_saved["total_eventos_nao_entregues"]
    != validacao_gold_saved["total_eventos_gold"]
):
    raise Exception("Erro: entregues + não entregues não fecha na Gold Delta.")

if chaves_duplicadas_gold_saved > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold Delta.")

if nulos_gold_saved > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold Delta.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_saved
    .select(
        col("ano_evento").cast("int").alias("ano_evento"),
        col("mes_evento").cast("int").alias("mes_evento"),
        col("evento_em_feriado").cast("int").alias("evento_em_feriado"),
        col("qtd_eventos").cast("int").alias("qtd_eventos"),
        col("qtd_eventos_entregues").cast("int").alias("qtd_eventos_entregues"),
        col("qtd_eventos_nao_entregues").cast("int").alias("qtd_eventos_nao_entregues"),
        col("taxa_sucesso_percentual").cast("decimal(10,2)").alias("taxa_sucesso_percentual"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")
df_gold_sql.printSchema()
display(df_gold_sql.orderBy("ano_evento", "mes_evento", "evento_em_feriado"))

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final do SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_final = df_final.count()

total_chaves_final = (
    df_final
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_final = total_linhas_final - total_chaves_final

validacao_final = (
    df_final
    .agg(
        spark_sum("qtd_eventos").alias("total_eventos_gold"),
        spark_sum("qtd_eventos_entregues").alias("total_eventos_entregues"),
        spark_sum("qtd_eventos_nao_entregues").alias("total_eventos_nao_entregues")
    )
    .collect()[0]
)

nulos_final = (
    df_final
    .filter(
        col("ano_evento").isNull() |
        col("mes_evento").isNull() |
        col("evento_em_feriado").isNull() |
        col("qtd_eventos").isNull() |
        col("qtd_eventos_entregues").isNull() |
        col("qtd_eventos_nao_entregues").isNull() |
        col("taxa_sucesso_percentual").isNull()
    )
    .count()
)

print(f"Total linhas tabela final: {total_linhas_final}")
print(f"Chaves duplicadas tabela final: {chaves_duplicadas_final}")
print(f"Total eventos fonte: {total_rastreamento}")
print(f"Total eventos tabela final: {validacao_final['total_eventos_gold']}")
print(f"Eventos entregues tabela final: {validacao_final['total_eventos_entregues']}")
print(f"Eventos não entregues tabela final: {validacao_final['total_eventos_nao_entregues']}")
print(f"Linhas com nulos principais tabela final: {nulos_final}")

if validacao_final["total_eventos_gold"] != total_rastreamento:
    raise Exception("Erro: total de eventos da tabela final não confere.")

if (
    validacao_final["total_eventos_entregues"] +
    validacao_final["total_eventos_nao_entregues"]
    != validacao_final["total_eventos_gold"]
):
    raise Exception("Erro: entregues + não entregues não fecha na tabela final.")

if chaves_duplicadas_final > 0:
    raise Exception("Erro: existem chaves duplicadas na tabela final.")

if nulos_final > 0:
    raise Exception("Erro: existem nulos nas colunas principais da tabela final.")

print("Validação OK: tabela final SQL Server gravada corretamente.")